## STEP 0: Import Data

In [21]:
import pandas as pd
import numpy as np
import pickle

# Load the pre-calculated features
with open('artist_features.pkl', 'rb') as f:
    features = pickle.load(f)

artist_stats_lookup = features['artist_stats_lookup']
top_15_collabs_list = features['top_15_collabs_list']
artist_popularity_lookup = features['artist_popularity_lookup']
global_mean_popularity = features['global_mean_popularity']

print("✓ Loaded pre-calculated features:")
print(f"  - artist_stats_lookup: {len(artist_stats_lookup):,} artists")
print(f"  - top_15_collabs_list: {len(top_15_collabs_list)} collaborators")
print(f"  - artist_popularity_lookup: {len(artist_popularity_lookup):,} artists")
print(f"  - global_mean_popularity: {global_mean_popularity:.2f}")


✓ Loaded pre-calculated features:
  - artist_stats_lookup: 1,469 artists
  - top_15_collabs_list: 15 collaborators
  - artist_popularity_lookup: 1,469 artists
  - global_mean_popularity: 67.67


## STEP 1: Temporal Features (Simple)
Extract from release_date column
Temperoal features compute for all train/val/test sets

In [22]:
# Load the aggregated dataset
agg_df = pd.read_csv('aggregated_dataset.csv', sep=';')
print(f"\n✓ Loaded aggregated dataset: {len(agg_df):,} rows")
print(f"  - Train: {(agg_df['split'] == 'train').sum():,} rows")
print(f"  - Validate: {(agg_df['split'] == 'val').sum():,} rows")
print(f"  - Test: {(agg_df['split'] == 'test').sum():,} rows")
print(agg_df.head())

# Convert first_appearance to datetime for ALL data
agg_df['first_appearance'] = pd.to_datetime(agg_df['first_appearance'])

# Extract temporal features from first_appearance (this is safe - no data leakage)
agg_df['release_year'] = agg_df['first_appearance'].dt.year
agg_df['release_quarter'] = agg_df['first_appearance'].dt.quarter
agg_df['release_month'] = agg_df['first_appearance'].dt.month
agg_df['release_day_of_week'] = agg_df['first_appearance'].dt.dayofweek
agg_df['is_friday_release'] = (agg_df['release_day_of_week'] == 4).astype(int)
agg_df['is_holiday_season'] = agg_df['release_month'].isin([11, 12]).astype(int)

print("\n✅ Temporal features created for train/validate/test!")

# Split into train, validate, and test
train_df = agg_df[agg_df['split'] == 'train'].copy()
val_df = agg_df[agg_df['split'] == 'val'].copy()
test_df = agg_df[agg_df['split'] == 'test'].copy()

print(f"\nTemporal Features Summary (Train Set):")
print(f"  - release_year: {train_df['release_year'].min()} to {train_df['release_year'].max()}")
print(f"  - release_quarter: {train_df['release_quarter'].value_counts().sort_index().to_dict()}")
print(f"  - is_friday_release: {train_df['is_friday_release'].sum():,} Friday releases ({train_df['is_friday_release'].mean()*100:.1f}%)")
print(f"  - is_holiday_season: {train_df['is_holiday_season'].sum():,} holiday releases ({train_df['is_holiday_season'].mean()*100:.1f}%)")

print("\nSample of temporal features (Train):")
print(train_df[['first_appearance', 'release_year', 'release_quarter', 'release_month', 
                'release_day_of_week', 'is_friday_release', 'is_holiday_season']].head(10))


✓ Loaded aggregated dataset: 9,161 rows
  - Train: 6,158 rows
  - Validate: 1,414 rows
  - Test: 1,589 rows
                       id  best_rank    avg_rank  total_weeks_charted  \
0  000xQL6tZNLJzIrtIgxqSl         40  101.500000                  118   
1  003VDDA7J3Xb2ZFlNx7nIZ        108  138.000000                    2   
2  003eoIwxETJujVWmNFMoZy         91  136.500000                   14   
3  003vvx7Niy0yvhvHt4a68B         73  168.069643                  560   
4  00B7TZ0Xawar6NZ00JFomN         61  109.285714                   14   

  first_appearance last_appearance                                 Title  \
0       2017-03-24      2017-09-12  Still Got Time (feat. PARTYNEXTDOOR)   
1       2020-02-07      2020-02-08                               YELL OH   
2       2018-06-15      2018-06-28                         Growing Pains   
3       2020-08-15      2023-01-01                        Mr. Brightside   
4       2018-04-06      2018-04-19   Best Life (feat. Chance The Rapper)

In [23]:
print(train_df.head(5))
print(val_df.head(5))
print(test_df.head(5))

#Check number songs in each set
print(f"Train set songs: {train_df['id'].nunique():,}")
print(f"Validation set songs: {val_df['id'].nunique():,}")
print(f"Test set songs: {test_df['id'].nunique():,}")


                       id  best_rank    avg_rank  total_weeks_charted  \
0  000xQL6tZNLJzIrtIgxqSl         40  101.500000                  118   
1  003VDDA7J3Xb2ZFlNx7nIZ        108  138.000000                    2   
2  003eoIwxETJujVWmNFMoZy         91  136.500000                   14   
3  003vvx7Niy0yvhvHt4a68B         73  168.069643                  560   
4  00B7TZ0Xawar6NZ00JFomN         61  109.285714                   14   

  first_appearance last_appearance                                 Title  \
0       2017-03-24      2017-09-12  Still Got Time (feat. PARTYNEXTDOOR)   
1       2020-02-07      2020-02-08                               YELL OH   
2       2018-06-15      2018-06-28                         Growing Pains   
3       2020-08-15      2023-01-01                        Mr. Brightside   
4       2018-04-06      2018-04-19   Best Life (feat. Chance The Rapper)   

         Artist  peak_score  longevity_score  ...  Speechiness Acousticness  \
0          ZAYN        80

## STEP 2: Artist Features
Use artist_stats_lookup to create 3 features
Extract primary artist for TRAIN and VAL only (not test - we'll handle test separately)


2A: Extract Primary Artist from 'Artists' Column

In [24]:
def extract_primary_artist(artists_string):
    """Extract first artist from comma-separated list"""
    artists = [a.strip() for a in str(artists_string).split(',')]
    return artists[0]  # First artist is primary

# Extract primary artist for train and val
train_df['primary_artist'] = train_df['Artist'].apply(extract_primary_artist)
val_df['primary_artist'] = val_df['Artist'].apply(extract_primary_artist)

print("✅ Primary artist extracted for train and val!")
print(f"\nTrain set unique artists: {train_df['primary_artist'].nunique():,}")
print(f"Val set unique artists: {val_df['primary_artist'].nunique():,}")

# Sample
print("\nSample (Train):")
print(train_df[['Artist', 'primary_artist']].head(10))



✅ Primary artist extracted for train and val!

Train set unique artists: 1,019
Val set unique artists: 487

Sample (Train):
         Artist primary_artist
0          ZAYN           ZAYN
1  Trippie Redd   Trippie Redd
2  Alessia Cara   Alessia Cara
3   The Killers    The Killers
4       Cardi B        Cardi B
5        Thalia         Thalia
6          Kygo           Kygo
7  Busta Rhymes   Busta Rhymes
8  Fall Out Boy   Fall Out Boy
9         DJ A1          DJ A1


In [25]:
print(train_df.head(5))
print(val_df.head(5))


                       id  best_rank    avg_rank  total_weeks_charted  \
0  000xQL6tZNLJzIrtIgxqSl         40  101.500000                  118   
1  003VDDA7J3Xb2ZFlNx7nIZ        108  138.000000                    2   
2  003eoIwxETJujVWmNFMoZy         91  136.500000                   14   
3  003vvx7Niy0yvhvHt4a68B         73  168.069643                  560   
4  00B7TZ0Xawar6NZ00JFomN         61  109.285714                   14   

  first_appearance last_appearance                                 Title  \
0       2017-03-24      2017-09-12  Still Got Time (feat. PARTYNEXTDOOR)   
1       2020-02-07      2020-02-08                               YELL OH   
2       2018-06-15      2018-06-28                         Growing Pains   
3       2020-08-15      2023-01-01                        Mr. Brightside   
4       2018-04-06      2018-04-19   Best Life (feat. Chance The Rapper)   

         Artist  peak_score  longevity_score  ...  Acousticness  \
0          ZAYN        80.5          

2B: Look Up Artist Statistics

In [26]:
# Calculate fallback values from training data (known artists only)
print("\n" + "=" * 80)
print("CALCULATING DATA-DRIVEN FALLBACK VALUES")
print("=" * 80)

# Get all artist stats from lookup (these are from training data only)
all_song_counts = [stats['song_count'] for stats in artist_stats_lookup.values()]
all_avg_ranks = [stats['avg_rank'] for stats in artist_stats_lookup.values()]
all_best_ranks = [stats['best_rank'] for stats in artist_stats_lookup.values()]

# Calculate fallback values
artist_fallback = {
    'song_count': 1,                              # Conservative - assume new artist
    'avg_rank': np.median(all_avg_ranks),         # Data-driven median
    'best_rank': np.median(all_best_ranks)        # Data-driven median
}

print(f"Fallback values calculated from {len(artist_stats_lookup):,} artists:")
print(f"  - song_count: {artist_fallback['song_count']}")
print(f"  - avg_rank: {artist_fallback['avg_rank']:.2f}")
print(f"  - best_rank: {artist_fallback['best_rank']:.2f}")


CALCULATING DATA-DRIVEN FALLBACK VALUES
Fallback values calculated from 1,469 artists:
  - song_count: 1
  - avg_rank: 133.31
  - best_rank: 72.00


In [27]:
def get_artist_features(primary_artist, artist_stats_lookup):
    """
    Look up artist statistics from pre-calculated lookup
    Returns: (song_count, avg_rank, best_rank)
    """
    if primary_artist in artist_stats_lookup:
        stats = artist_stats_lookup[primary_artist]
        return (
            stats['song_count'],
            stats['avg_rank'],
            stats['best_rank']
        )
    else:
        # Artist not found - use data-driven fallback
        return (
            artist_fallback['song_count'],
            artist_fallback['avg_rank'],
            artist_fallback['best_rank']
        )
# Apply to train set
print("\nProcessing train set...")
artist_features_train = train_df['primary_artist'].apply(
    lambda x: get_artist_features(x, artist_stats_lookup)
)

# Split into separate columns for train
train_df['artist_song_count'] = artist_features_train.apply(lambda x: x[0])
train_df['artist_avg_rank'] = artist_features_train.apply(lambda x: x[1])
train_df['artist_best_rank'] = artist_features_train.apply(lambda x: x[2])

# Apply to validation set
print("Processing validation set...")
artist_features_val = val_df['primary_artist'].apply(
    lambda x: get_artist_features(x, artist_stats_lookup)
)

# Split into separate columns for val
val_df['artist_song_count'] = artist_features_val.apply(lambda x: x[0])
val_df['artist_avg_rank'] = artist_features_val.apply(lambda x: x[1])
val_df['artist_best_rank'] = artist_features_val.apply(lambda x: x[2])

print("\n✅ Artist features created for train and val with data-driven fallback!")

# Summary statistics
print("\n📊 Artist Features Summary (Train Set):")
print(train_df[['artist_song_count', 'artist_avg_rank', 'artist_best_rank']].describe())

print("\n📊 Artist Features Summary (Val Set):")
print(val_df[['artist_song_count', 'artist_avg_rank', 'artist_best_rank']].describe())

# Check for fallback usage (updated condition)
train_fallback = (train_df['artist_song_count'] == artist_fallback['song_count']) & \
                 (train_df['artist_avg_rank'] == artist_fallback['avg_rank'])
val_fallback = (val_df['artist_song_count'] == artist_fallback['song_count']) & \
               (val_df['artist_avg_rank'] == artist_fallback['avg_rank'])

print(f"\n🔍 Fallback Usage:")
print(f"  - Train: {train_fallback.sum():,} artists not found ({train_fallback.mean()*100:.1f}%)")
print(f"  - Val: {val_fallback.sum():,} artists not found ({val_fallback.mean()*100:.1f}%)")

# Sample results
print("\n📋 Sample Results (Train):")
print(train_df[['primary_artist', 'artist_song_count', 'artist_avg_rank', 'artist_best_rank']].head(10))


Processing train set...
Processing validation set...

✅ Artist features created for train and val with data-driven fallback!

📊 Artist Features Summary (Train Set):
       artist_song_count  artist_avg_rank  artist_best_rank
count        6158.000000      6158.000000       6158.000000
mean           31.221988       113.201231         27.367002
std            30.534962        26.359987         40.732378
min             1.000000        16.000000          1.000000
25%             8.000000        95.060000          2.000000
50%            21.000000       105.670000          8.000000
75%            44.000000       128.250000         33.000000
max           123.000000       200.000000        200.000000

📊 Artist Features Summary (Val Set):
       artist_song_count  artist_avg_rank  artist_best_rank
count        1414.000000      1414.000000        1414.00000
mean           23.218529       117.219441          36.91372
std            30.584278        26.443321          40.12794
min             

In [28]:
# Verify fallback values are correctly applied
print("\n" + "=" * 100)
print("VERIFICATION: Confirming Fallback Values Applied to Unknown Artists")
print("=" * 100)

# Identify unknown artists in train set
train_unknown = train_df[train_fallback].copy()
print(f"\n📋 TRAIN SET - Unknown Artists:")
print(f"Total songs with unknown artists: {len(train_unknown)}")

if len(train_unknown) > 0:
    # Check if all have exact fallback values
    all_correct_train = (
        (train_unknown['artist_song_count'] == artist_fallback['song_count']).all() and
        (train_unknown['artist_avg_rank'] == artist_fallback['avg_rank']).all() and
        (train_unknown['artist_best_rank'] == artist_fallback['best_rank']).all()
    )
    print(f"✅ All using fallback ({artist_fallback['song_count']}, {artist_fallback['avg_rank']:.2f}, {artist_fallback['best_rank']:.2f}): {all_correct_train}")
    
    if all_correct_train:
        print("\n✅ VERIFIED: All unknown artists in train set have correct fallback values")
    else:
        print("\n⚠️ WARNING: Some unknown artists don't have correct fallback values!")
    
    print("\nUnknown artists in train:")
    print(train_unknown[['primary_artist', 'artist_song_count', 'artist_avg_rank', 'artist_best_rank']])
else:
    print("✅ No unknown artists in train set (expected - all should be in lookup)")

# Identify unknown artists in validation set
val_unknown = val_df[val_fallback].copy()
print(f"\n📋 VALIDATION SET - Unknown Artists:")
print(f"Total songs with unknown artists: {len(val_unknown)} ({len(val_unknown)/len(val_df)*100:.1f}% of val set)")

if len(val_unknown) > 0:
    # Check if all have exact fallback values
    all_correct_val = (
        (val_unknown['artist_song_count'] == artist_fallback['song_count']).all() and
        (val_unknown['artist_avg_rank'] == artist_fallback['avg_rank']).all() and
        (val_unknown['artist_best_rank'] == artist_fallback['best_rank']).all()
    )
    print(f"✅ All using fallback ({artist_fallback['song_count']}, {artist_fallback['avg_rank']:.2f}, {artist_fallback['best_rank']:.2f}): {all_correct_val}")
    
    if all_correct_val:
        print("\n✅ VERIFIED: All unknown artists in validation set have correct fallback values")
    else:
        print("\n⚠️ WARNING: Some unknown artists don't have correct fallback values!")
    
    print(f"\nFirst 10 unknown artists in validation:")
    print(val_unknown[['primary_artist', 'artist_song_count', 'artist_avg_rank', 'artist_best_rank']].head(10))
    
    # Show unique unknown artists
    unique_unknown = val_unknown['primary_artist'].nunique()
    print(f"\n📊 Unique unknown artists in validation: {unique_unknown}")

print("\n" + "=" * 100)
print("FINAL VERIFICATION SUMMARY:")
print("=" * 100)
print(f"✅ Fallback values: song_count={artist_fallback['song_count']}, avg_rank={artist_fallback['avg_rank']:.2f}, best_rank={artist_fallback['best_rank']:.2f}")
print(f"✅ Train: {len(train_unknown)} songs using fallback")
print(f"✅ Val: {len(val_unknown)} songs using fallback")
print(f"✅ All fallback values correctly applied: {all_correct_train if len(train_unknown) > 0 else True} (train), {all_correct_val} (val)")


VERIFICATION: Confirming Fallback Values Applied to Unknown Artists

📋 TRAIN SET - Unknown Artists:
Total songs with unknown artists: 2
✅ All using fallback (1, 133.31, 72.00): True

✅ VERIFIED: All unknown artists in train set have correct fallback values

Unknown artists in train:
          primary_artist  artist_song_count  artist_avg_rank  artist_best_rank
325   Humberto & Ronaldo                  1           133.31              72.0
3498           Burna Boy                  1           133.31              72.0

📋 VALIDATION SET - Unknown Artists:
Total songs with unknown artists: 289 (20.4% of val set)
✅ All using fallback (1, 133.31, 72.00): True

✅ VERIFIED: All unknown artists in validation set have correct fallback values

First 10 unknown artists in validation:
        primary_artist  artist_song_count  artist_avg_rank  artist_best_rank
6163       sangiovanni                  1           133.31              72.0
6165       Kevin Gates                  1           133.31     

## STEP 3: Collaboration Features
Use all collab lookups to create 19 features

3A: Basic Count - num_collab_artists

In [29]:
def count_collaborators(artists_string):
    """Count number of collaborators (total artists - 1 for primary)"""
    artists = [a.strip() for a in str(artists_string).split(',')]
    return len(artists) - 1  # Subtract primary artist

# Apply to train set
train_df['num_collab_artists'] = train_df['Artist'].apply(count_collaborators)

# Apply to validation set
val_df['num_collab_artists'] = val_df['Artist'].apply(count_collaborators)

print("✅ Basic collaboration count created for train and val!")

# Summary statistics
print("\n📊 Collaboration Count Summary:")
print(f"Train Set:")
print(f"  - Songs with no collaborators: {(train_df['num_collab_artists'] == 0).sum():,} ({(train_df['num_collab_artists'] == 0).mean()*100:.1f}%)")
print(f"  - Songs with collaborators: {(train_df['num_collab_artists'] > 0).sum():,} ({(train_df['num_collab_artists'] > 0).mean()*100:.1f}%)")
print(f"  - Max collaborators: {train_df['num_collab_artists'].max()}")
print(f"  - Mean collaborators: {train_df['num_collab_artists'].mean():.2f}")

print(f"\nValidation Set:")
print(f"  - Songs with no collaborators: {(val_df['num_collab_artists'] == 0).sum():,} ({(val_df['num_collab_artists'] == 0).mean()*100:.1f}%)")
print(f"  - Songs with collaborators: {(val_df['num_collab_artists'] > 0).sum():,} ({(val_df['num_collab_artists'] > 0).mean()*100:.1f}%)")
print(f"  - Max collaborators: {val_df['num_collab_artists'].max()}")
print(f"  - Mean collaborators: {val_df['num_collab_artists'].mean():.2f}")

# Sample results
print("\n📋 Sample Results (Train):")
print(train_df[['Artist', 'num_collab_artists']].head(10))

# Distribution
print("\n📊 Distribution of Collaboration Counts (Train):")
print(train_df['num_collab_artists'].value_counts().sort_index().head(10))

✅ Basic collaboration count created for train and val!

📊 Collaboration Count Summary:
Train Set:
  - Songs with no collaborators: 6,134 (99.6%)
  - Songs with collaborators: 24 (0.4%)
  - Max collaborators: 1
  - Mean collaborators: 0.00

Validation Set:
  - Songs with no collaborators: 1,397 (98.8%)
  - Songs with collaborators: 17 (1.2%)
  - Max collaborators: 1
  - Mean collaborators: 0.01

📋 Sample Results (Train):
         Artist  num_collab_artists
0          ZAYN                   0
1  Trippie Redd                   0
2  Alessia Cara                   0
3   The Killers                   0
4       Cardi B                   0
5        Thalia                   0
6          Kygo                   0
7  Busta Rhymes                   0
8  Fall Out Boy                   0
9         DJ A1                   0

📊 Distribution of Collaboration Counts (Train):
num_collab_artists
0    6134
1      24
Name: count, dtype: int64


3B: Binary Features - Top 15 Collaborators

In [30]:
#Remove

3C: Numeric Feature - collab_artist_avg_popularity

In [31]:
def get_collab_avg_popularity(artists_string, artist_popularity_lookup, global_mean):
    """
    Calculate average popularity of collaborators
    """
    # Parse artists
    artists = [a.strip() for a in str(artists_string).split(',')]
    
    # If only one artist (no collaborators)
    if len(artists) == 1:
        return 0
    
    # Get collaborators (all except first)
    collaborators = artists[1:]
    
    # Get popularity scores for each collaborator
    collab_scores = []
    for collab in collaborators:
        if collab in artist_popularity_lookup:
            score = artist_popularity_lookup[collab]
        else:
            # Unknown collaborator - use global mean
            score = global_mean
        collab_scores.append(score)
    
    # Return average
    return np.mean(collab_scores) if collab_scores else 0

# Apply to train set
print("Processing train set...")
train_df['collab_artist_avg_popularity'] = train_df['Artist'].apply(
    lambda x: get_collab_avg_popularity(x, artist_popularity_lookup, global_mean_popularity)
)

# Apply to validation set
print("Processing validation set...")
val_df['collab_artist_avg_popularity'] = val_df['Artist'].apply(
    lambda x: get_collab_avg_popularity(x, artist_popularity_lookup, global_mean_popularity)
)

print("\n✅ Collaborator average popularity created for train and val!")

# Summary statistics
print("\n📊 Collaborator Popularity Summary:")
print(f"Train Set:")
print(f"  - Songs with no collaborators (0.0): {(train_df['collab_artist_avg_popularity'] == 0).sum():,}")
print(f"  - Songs with collaborators: {(train_df['collab_artist_avg_popularity'] > 0).sum():,}")
print(f"  - Mean popularity (all songs): {train_df['collab_artist_avg_popularity'].mean():.2f}")
print(f"  - Mean popularity (songs with collabs): {train_df[train_df['collab_artist_avg_popularity'] > 0]['collab_artist_avg_popularity'].mean():.2f}")
print(f"  - Min/Max: {train_df['collab_artist_avg_popularity'].min():.2f} / {train_df['collab_artist_avg_popularity'].max():.2f}")

print(f"\nValidation Set:")
print(f"  - Songs with no collaborators (0.0): {(val_df['collab_artist_avg_popularity'] == 0).sum():,}")
print(f"  - Songs with collaborators: {(val_df['collab_artist_avg_popularity'] > 0).sum():,}")
print(f"  - Mean popularity (all songs): {val_df['collab_artist_avg_popularity'].mean():.2f}")
print(f"  - Mean popularity (songs with collabs): {val_df[val_df['collab_artist_avg_popularity'] > 0]['collab_artist_avg_popularity'].mean():.2f}")
print(f"  - Min/Max: {val_df['collab_artist_avg_popularity'].min():.2f} / {val_df['collab_artist_avg_popularity'].max():.2f}")

# Sample results
print("\n📋 Sample Results (Train):")
print(train_df[['Artist', 'num_collab_artists', 'collab_artist_avg_popularity']].head(10))

# Detailed statistics for songs with collaborators
print("\n📊 Detailed Statistics (Train - Songs with Collaborators):")
songs_with_collabs = train_df[train_df['num_collab_artists'] > 0]
print(songs_with_collabs['collab_artist_avg_popularity'].describe())

Processing train set...
Processing validation set...

✅ Collaborator average popularity created for train and val!

📊 Collaborator Popularity Summary:
Train Set:
  - Songs with no collaborators (0.0): 6,134
  - Songs with collaborators: 24
  - Mean popularity (all songs): 0.36
  - Mean popularity (songs with collabs): 92.71
  - Min/Max: 0.00 / 92.71

Validation Set:
  - Songs with no collaborators (0.0): 1,397
  - Songs with collaborators: 17
  - Mean popularity (all songs): 1.11
  - Mean popularity (songs with collabs): 92.71
  - Min/Max: 0.00 / 92.71

📋 Sample Results (Train):
         Artist  num_collab_artists  collab_artist_avg_popularity
0          ZAYN                   0                           0.0
1  Trippie Redd                   0                           0.0
2  Alessia Cara                   0                           0.0
3   The Killers                   0                           0.0
4       Cardi B                   0                           0.0
5        Thalia   

3D: Numeric Features - Frequency Statistics

In [32]:
# First, calculate collaborator frequencies from training data
print("=" * 100)
print("CALCULATING COLLABORATOR FREQUENCIES FROM TRAINING DATA")
print("=" * 100)

# Count how many times each artist appears as a collaborator in training set
from collections import Counter

collaborator_frequency = Counter()

# Count collaborators in training data only
for artists_string in train_df['Artist']:
    artists = [a.strip() for a in str(artists_string).split(',')]
    # Skip primary artist (index 0), only count collaborators
    if len(artists) > 1:
        collaborators = artists[1:]
        for collab in collaborators:
            collaborator_frequency[collab] += 1

# Convert to regular dict
collaborator_frequency = dict(collaborator_frequency)

print(f"\n✓ Calculated frequencies for {len(collaborator_frequency):,} unique collaborators")
print(f"  - Total collaborations counted: {sum(collaborator_frequency.values()):,}")
print(f"\nTop 10 most frequent collaborators:")
top_collabs = sorted(collaborator_frequency.items(), key=lambda x: x[1], reverse=True)[:10]
for collab, freq in top_collabs:
    print(f"  - {collab}: {freq} appearances")

# Define function to extract frequency features
def get_collab_frequency_features(artists_string, collaborator_frequency):
    """
    Get max and avg frequency of collaborators
    """
    # Parse artists
    artists = [a.strip() for a in str(artists_string).split(',')]
    
    # If only one artist (no collaborators)
    if len(artists) == 1:
        return (0, 0)
    
    # Get collaborators
    collaborators = artists[1:]
    
    # Get frequencies
    frequencies = []
    for collab in collaborators:
        if collab in collaborator_frequency:
            frequencies.append(collaborator_frequency[collab])
        else:
            frequencies.append(0)  # Unknown collaborator
    
    if frequencies:
        return (max(frequencies), np.mean(frequencies))
    else:
        return (0, 0)

# Apply to train set
print("\n" + "=" * 100)
print("APPLYING FREQUENCY FEATURES TO TRAIN SET")
print("=" * 100)

freq_features_train = train_df['Artist'].apply(
    lambda x: get_collab_frequency_features(x, collaborator_frequency)
)

train_df['collab_max_frequency'] = freq_features_train.apply(lambda x: x[0])
train_df['collab_avg_frequency'] = freq_features_train.apply(lambda x: x[1])

# Apply to validation set
print("\nAPPLYING FREQUENCY FEATURES TO VALIDATION SET")

freq_features_val = val_df['Artist'].apply(
    lambda x: get_collab_frequency_features(x, collaborator_frequency)
)

val_df['collab_max_frequency'] = freq_features_val.apply(lambda x: x[0])
val_df['collab_avg_frequency'] = freq_features_val.apply(lambda x: x[1])

print("\n✅ Frequency features created for train and val!")

# Summary statistics
print("\n" + "=" * 100)
print("📊 COLLABORATOR FREQUENCY FEATURES SUMMARY")
print("=" * 100)

print(f"\nTrain Set:")
print(f"  - Songs with no collaborators (0, 0): {((train_df['collab_max_frequency'] == 0) & (train_df['collab_avg_frequency'] == 0)).sum():,}")
print(f"  - Songs with collaborators: {(train_df['collab_max_frequency'] > 0).sum():,}")
print(f"\n  Max Frequency Statistics:")
print(f"    - Mean: {train_df['collab_max_frequency'].mean():.2f}")
print(f"    - Median: {train_df['collab_max_frequency'].median():.2f}")
print(f"    - Min/Max: {train_df['collab_max_frequency'].min()} / {train_df['collab_max_frequency'].max()}")
print(f"\n  Avg Frequency Statistics:")
print(f"    - Mean: {train_df['collab_avg_frequency'].mean():.2f}")
print(f"    - Median: {train_df['collab_avg_frequency'].median():.2f}")
print(f"    - Min/Max: {train_df['collab_avg_frequency'].min():.2f} / {train_df['collab_avg_frequency'].max():.2f}")

print(f"\nValidation Set:")
print(f"  - Songs with no collaborators (0, 0): {((val_df['collab_max_frequency'] == 0) & (val_df['collab_avg_frequency'] == 0)).sum():,}")
print(f"  - Songs with collaborators: {(val_df['collab_max_frequency'] > 0).sum():,}")
print(f"\n  Max Frequency Statistics:")
print(f"    - Mean: {val_df['collab_max_frequency'].mean():.2f}")
print(f"    - Median: {val_df['collab_max_frequency'].median():.2f}")
print(f"    - Min/Max: {val_df['collab_max_frequency'].min()} / {val_df['collab_max_frequency'].max()}")
print(f"\n  Avg Frequency Statistics:")
print(f"    - Mean: {val_df['collab_avg_frequency'].mean():.2f}")
print(f"    - Median: {val_df['collab_avg_frequency'].median():.2f}")
print(f"    - Min/Max: {val_df['collab_avg_frequency'].min():.2f} / {val_df['collab_avg_frequency'].max():.2f}")

# Sample results
print("\n📋 Sample Results (Train - Songs with Collaborators):")
train_with_collabs = train_df[train_df['num_collab_artists'] > 0][['Artist', 'num_collab_artists', 'collab_max_frequency', 'collab_avg_frequency']].head(10)
print(train_with_collabs)

# Check for unknown collaborators (frequency = 0)
train_unknown_collabs = (train_df['num_collab_artists'] > 0) & (train_df['collab_max_frequency'] == 0)
val_unknown_collabs = (val_df['num_collab_artists'] > 0) & (val_df['collab_max_frequency'] == 0)

print(f"\n🔍 Unknown Collaborators (not in training data):")
print(f"  - Train: {train_unknown_collabs.sum()} songs have unknown collaborators")
print(f"  - Val: {val_unknown_collabs.sum()} songs have unknown collaborators")

CALCULATING COLLABORATOR FREQUENCIES FROM TRAINING DATA

✓ Calculated frequencies for 1 unique collaborators
  - Total collaborations counted: 24

Top 10 most frequent collaborators:
  - The Creator: 24 appearances

APPLYING FREQUENCY FEATURES TO TRAIN SET

APPLYING FREQUENCY FEATURES TO VALIDATION SET

✅ Frequency features created for train and val!

📊 COLLABORATOR FREQUENCY FEATURES SUMMARY

Train Set:
  - Songs with no collaborators (0, 0): 6,134
  - Songs with collaborators: 24

  Max Frequency Statistics:
    - Mean: 0.09
    - Median: 0.00
    - Min/Max: 0 / 24

  Avg Frequency Statistics:
    - Mean: 0.09
    - Median: 0.00
    - Min/Max: 0.00 / 24.00

Validation Set:
  - Songs with no collaborators (0, 0): 1,397
  - Songs with collaborators: 17

  Max Frequency Statistics:
    - Mean: 0.29
    - Median: 0.00
    - Min/Max: 0 / 24

  Avg Frequency Statistics:
    - Mean: 0.29
    - Median: 0.00
    - Min/Max: 0.00 / 24.00

📋 Sample Results (Train - Songs with Collaborators):
   

## STEP 4: Verify All Features Created

In [33]:
# Define expected feature columns
print("\n📋 EXPECTED FEATURES:")
print("=" * 100)

# Audio features (already in aggregated_dataset.csv)
audio_features = [
    'Danceability', 'Energy', 'Loudness_norm', 'Speechiness', 
    'Acousticness', 'Instrumentalness', 'Valence'
]
print(f"\n1️⃣  Audio Features ({len(audio_features)}): {audio_features}")

# Temporal features (created in Step 1)
temporal_features = [
    'release_year', 'release_quarter', 'release_month', 
    'is_friday_release', 'is_holiday_season'
]
print(f"\n2️⃣  Temporal Features ({len(temporal_features)}): {temporal_features}")

# Artist features (created in Step 2)
artist_features = [
    'artist_song_count', 'artist_avg_rank', 'artist_best_rank'
]
print(f"\n3️⃣  Artist Features ({len(artist_features)}): {artist_features}")

# Collaboration features (created in Step 3)
collab_features = [
    'num_collab_artists',
    'collab_artist_avg_popularity',
    'collab_max_frequency',
    'collab_avg_frequency'
]
print(f"\n4️⃣  Collaboration Features ({len(collab_features)}): {collab_features}")

# All feature columns
all_feature_columns = audio_features + temporal_features + artist_features + collab_features
print(f"\n📊 TOTAL FEATURES: {len(all_feature_columns)}")
print("=" * 100)

# ====================================================================================================
# CHECK TRAIN SET
# ====================================================================================================
print("\n" + "=" * 100)
print("CHECKING TRAIN SET")
print("=" * 100)

print(f"\n✓ Shape: {train_df.shape}")
print(f"✓ Total columns: {len(train_df.columns)}")

# Check which features are present
print(f"\n📋 Feature Presence Check:")
missing_features = []
present_features = []

for feature in all_feature_columns:
    if feature in train_df.columns:
        present_features.append(feature)
    else:
        missing_features.append(feature)

print(f"  ✅ Present: {len(present_features)}/{len(all_feature_columns)}")
if missing_features:
    print(f"  ❌ Missing: {len(missing_features)} features")
    print(f"     {missing_features}")
else:
    print(f"  ✅ All features present!")

# Check for missing values
print(f"\n🔍 Missing Values Check:")
missing_counts = train_df[all_feature_columns].isna().sum()
if missing_counts.sum() == 0:
    print("  ✅ No missing values in any feature!")
else:
    print(f"  ⚠️  Found missing values:")
    print(missing_counts[missing_counts > 0])

# Summary statistics
print(f"\n📊 Feature Summary Statistics (Train):")
print(train_df[all_feature_columns].describe())

# ====================================================================================================
# CHECK VALIDATION SET
# ====================================================================================================
print("\n" + "=" * 100)
print("CHECKING VALIDATION SET")
print("=" * 100)

print(f"\n✓ Shape: {val_df.shape}")
print(f"✓ Total columns: {len(val_df.columns)}")

# Check which features are present
print(f"\n📋 Feature Presence Check:")
missing_features_val = []
present_features_val = []

for feature in all_feature_columns:
    if feature in val_df.columns:
        present_features_val.append(feature)
    else:
        missing_features_val.append(feature)

print(f"  ✅ Present: {len(present_features_val)}/{len(all_feature_columns)}")
if missing_features_val:
    print(f"  ❌ Missing: {len(missing_features_val)} features")
    print(f"     {missing_features_val}")
else:
    print(f"  ✅ All features present!")

# Check for missing values
print(f"\n🔍 Missing Values Check:")
missing_counts_val = val_df[all_feature_columns].isna().sum()
if missing_counts_val.sum() == 0:
    print("  ✅ No missing values in any feature!")
else:
    print(f"  ⚠️  Found missing values:")
    print(missing_counts_val[missing_counts_val > 0])

# ====================================================================================================
# CHECK TEST SET (should only have temporal features)
# ====================================================================================================
print("\n" + "=" * 100)
print("CHECKING TEST SET (Should only have Audio + Temporal features)")
print("=" * 100)

print(f"\n✓ Shape: {test_df.shape}")
print(f"✓ Total columns: {len(test_df.columns)}")

# Test should only have audio + temporal features (no artist/collab features)
expected_test_features = audio_features + temporal_features

print(f"\n📋 Expected Features in Test: {len(expected_test_features)}")
print(f"  - Audio features: {len(audio_features)}")
print(f"  - Temporal features: {len(temporal_features)}")

# Check presence
present_test = [f for f in expected_test_features if f in test_df.columns]
missing_test = [f for f in expected_test_features if f not in test_df.columns]

print(f"\n✅ Present: {len(present_test)}/{len(expected_test_features)}")
if missing_test:
    print(f"❌ Missing: {missing_test}")

# Check that artist/collab features are NOT in test (as expected)
artist_collab_features = artist_features + collab_features
present_artist_collab = [f for f in artist_collab_features if f in test_df.columns]

if len(present_artist_collab) == 0:
    print(f"\n✅ Correctly NO artist/collaboration features in test set")
else:
    print(f"\n⚠️  WARNING: Found {len(present_artist_collab)} artist/collab features in test:")
    print(f"   {present_artist_collab}")

# ====================================================================================================
# FINAL SUMMARY
# ====================================================================================================
print("\n" + "=" * 100)
print("FINAL VERIFICATION SUMMARY")
print("=" * 100)

print(f"\n✅ TRAIN SET:")
print(f"   - Shape: {train_df.shape}")
print(f"   - Features: {len(present_features)}/{len(all_feature_columns)}")
print(f"   - Missing values: {missing_counts.sum()}")

print(f"\n✅ VALIDATION SET:")
print(f"   - Shape: {val_df.shape}")
print(f"   - Features: {len(present_features_val)}/{len(all_feature_columns)}")
print(f"   - Missing values: {missing_counts_val.sum()}")

print(f"\n✅ TEST SET:")
print(f"   - Shape: {test_df.shape}")
print(f"   - Features: {len(present_test)}/{len(expected_test_features)} (audio + temporal only)")
print(f"   - Artist/Collab features: {len(present_artist_collab)} (should be 0)")

if len(missing_features) == 0 and len(missing_features_val) == 0 and missing_counts.sum() == 0:
    print("\n" + "🎉" * 50)
    print("✅ ALL FEATURES SUCCESSFULLY CREATED!")
    print("✅ NO MISSING VALUES!")
    print("✅ READY FOR MODELING!")
    print("🎉" * 50)


📋 EXPECTED FEATURES:

1️⃣  Audio Features (7): ['Danceability', 'Energy', 'Loudness_norm', 'Speechiness', 'Acousticness', 'Instrumentalness', 'Valence']

2️⃣  Temporal Features (5): ['release_year', 'release_quarter', 'release_month', 'is_friday_release', 'is_holiday_season']

3️⃣  Artist Features (3): ['artist_song_count', 'artist_avg_rank', 'artist_best_rank']

4️⃣  Collaboration Features (4): ['num_collab_artists', 'collab_artist_avg_popularity', 'collab_max_frequency', 'collab_avg_frequency']

📊 TOTAL FEATURES: 19

CHECKING TRAIN SET

✓ Shape: (6158, 34)
✓ Total columns: 34

📋 Feature Presence Check:
  ✅ Present: 19/19
  ✅ All features present!

🔍 Missing Values Check:
  ✅ No missing values in any feature!

📊 Feature Summary Statistics (Train):
       Danceability       Energy  Loudness_norm  Speechiness  Acousticness  \
count   6158.000000  6158.000000    6158.000000  6158.000000   6158.000000   
mean       0.679738     0.635094       0.812788     0.133995      0.222857   
std   

## STEP 5: Create Final Feature Matrix

In [34]:
# Audio features (7 features)
# Note: Will check actual column names in the data
audio_features_cols = [
    'Danceability', 'Energy', 'Loudness_norm', 'Speechiness',
    'Acousticness', 'Instrumentalness', 'Valence'
]


# Temporal features (5 features)
temporal_features_cols = [
    'release_year', 'release_quarter', 'release_month',
    'is_friday_release', 'is_holiday_season'
]


# Artist features (3 features)
artist_features_cols = [
    'artist_song_count', 'artist_avg_rank', 'artist_best_rank'
]


# Collaboration features (4 features)
collab_features_cols = [
    'num_collab_artists',
    'collab_artist_avg_popularity',
    'collab_max_frequency',
    'collab_avg_frequency'
]


# Combine all features
all_features = (audio_features_cols + 
                temporal_features_cols + 
                artist_features_cols + 
                collab_features_cols)



In [35]:
# Check train set
print("\n📋 Train Set - Feature Availability:")
available_features_train = [f for f in all_features if f in train_df.columns]
missing_features_train = [f for f in all_features if f not in train_df.columns]

print(f"  ✅ Available: {len(available_features_train)}/{len(all_features)}")
if missing_features_train:
    print(f"  ⚠️  Missing: {len(missing_features_train)} features")
    print(f"     {missing_features_train}")

# Check validation set
print("\n📋 Validation Set - Feature Availability:")
available_features_val = [f for f in all_features if f in val_df.columns]
missing_features_val = [f for f in all_features if f not in val_df.columns]

print(f"  ✅ Available: {len(available_features_val)}/{len(all_features)}")
if missing_features_val:
    print(f"  ⚠️  Missing: {len(missing_features_val)} features")
    print(f"     {missing_features_val}")

# Use only available features
features_to_use = available_features_train
print(f"\n✅ Will use {len(features_to_use)} available features for modeling")



📋 Train Set - Feature Availability:
  ✅ Available: 19/19

📋 Validation Set - Feature Availability:
  ✅ Available: 19/19

✅ Will use 19 available features for modeling


In [36]:
# TRAIN SET
print("\n🔧 Creating Train Set matrices...")
X_train = train_df[features_to_use].copy()
y_train = train_df['popularity_label'].copy()

print(f"\n✅ Train Set Created:")
print(f"   - X_train shape: {X_train.shape}")
print(f"   - y_train shape: {y_train.shape}")

# VALIDATION SET
print("\n🔧 Creating Validation Set matrices...")
X_val = val_df[features_to_use].copy()
y_val = val_df['popularity_label'].copy()

print(f"\n✅ Validation Set Created:")
print(f"   - X_val shape: {X_val.shape}")
print(f"   - y_val shape: {y_val.shape}")




🔧 Creating Train Set matrices...

✅ Train Set Created:
   - X_train shape: (6158, 19)
   - y_train shape: (6158,)

🔧 Creating Validation Set matrices...

✅ Validation Set Created:
   - X_val shape: (1414, 19)
   - y_val shape: (1414,)


In [38]:
# Check for missing values
print("\n🔍 Missing Values Check:")
train_missing = X_train.isna().sum().sum()
val_missing = X_val.isna().sum().sum()
y_train_missing = y_train.isna().sum()
y_val_missing = y_val.isna().sum()

print(f"   - X_train: {train_missing} missing values")
print(f"   - X_val: {val_missing} missing values")
print(f"   - y_train: {y_train_missing} missing values")
print(f"   - y_val: {y_val_missing} missing values")

if train_missing > 0:
    print("\n⚠️  Features with missing values in X_train:")
    print(X_train.isna().sum()[X_train.isna().sum() > 0])

if val_missing > 0:
    print("\n⚠️  Features with missing values in X_val:")
    print(X_val.isna().sum()[X_val.isna().sum() > 0])

# Check data types
print("\n📊 Data Types (X_train):")
print(X_train.dtypes.value_counts())

# Summary statistics
print("\n📊 Summary Statistics (X_train):")
print(X_train.describe())




🔍 Missing Values Check:
   - X_train: 0 missing values
   - X_val: 0 missing values
   - y_train: 0 missing values
   - y_val: 0 missing values

📊 Data Types (X_train):
float64    11
int64       5
int32       3
Name: count, dtype: int64

📊 Summary Statistics (X_train):
       Danceability       Energy  Loudness_norm  Speechiness  Acousticness  \
count   6158.000000  6158.000000    6158.000000  6158.000000   6158.000000   
mean       0.679738     0.635094       0.812788     0.133995      0.222857   
std        0.147769     0.169185       0.076126     0.121321      0.242651   
min        0.073000     0.005000       0.000000     0.023000      0.000000   
25%        0.589000     0.533000       0.780319     0.046000      0.039000   
50%        0.696000     0.649000       0.824133     0.081000      0.131000   
75%        0.787000     0.759000       0.861943     0.191000      0.324750   
max        0.980000     0.996000       1.000000     0.966000      0.994000   

       Instrumentalness   

In [40]:
# Save as pickle file
print("\n💾 Saving feature matrices to pickle file...")
import pickle

feature_data = {
    'X_train': X_train,
    'X_val': X_val,
    'y_train': y_train,
    'y_val': y_val,
    'feature_names': features_to_use,
    'audio_features': [f for f in audio_features_cols if f in features_to_use],
    'temporal_features': [f for f in temporal_features_cols if f in features_to_use],
    'artist_features': [f for f in artist_features_cols if f in features_to_use],
    'collab_features': [f for f in collab_features_cols if f in features_to_use]
}

with open('feature_matrices.pkl', 'wb') as f:
    pickle.dump(feature_data, f)

print("   ✅ Saved to: feature_matrices.pkl")

# Also save as CSV for easy inspection
print("\n💾 Saving as CSV files...")
X_train.to_csv('X_train.csv', index=False)
X_val.to_csv('X_val.csv', index=False)
y_train.to_csv('y_train.csv', index=False, header=['popularity_label'])  # Changed header
y_val.to_csv('y_val.csv', index=False, header=['popularity_label'])

print("   ✅ Saved CSV files:")
print("      - X_train.csv")
print("      - X_val.csv")
print("      - y_train.csv")
print("      - y_val.csv")




💾 Saving feature matrices to pickle file...
   ✅ Saved to: feature_matrices.pkl

💾 Saving as CSV files...
   ✅ Saved CSV files:
      - X_train.csv
      - X_val.csv
      - y_train.csv
      - y_val.csv


In [ ]:
print("\n" + "=" * 100)
print("FINAL SUMMARY - FEATURE MATRICES READY FOR MODELING!")
print("=" * 100)

print(f"\n✅ TRAIN SET:")
print(f"   - X_train: {X_train.shape} (rows × features)")
print(f"   - y_train: {y_train.shape} (rows,)")
print(f"   - Missing values: {train_missing}")

print(f"\n✅ VALIDATION SET:")
print(f"   - X_val: {X_val.shape} (rows × features)")
print(f"   - y_val: {y_val.shape} (rows,)")
print(f"   - Missing values: {val_missing}")

print(f"\n📊 FEATURE BREAKDOWN:")
print(f"   - Audio features: {len([f for f in audio_features_cols if f in features_to_use])}/{len(audio_features_cols)}")
print(f"   - Temporal features: {len([f for f in temporal_features_cols if f in features_to_use])}/{len(temporal_features_cols)}")
print(f"   - Artist features: {len([f for f in artist_features_cols if f in features_to_use])}/{len(artist_features_cols)}")
print(f"   - Collaboration features: {len([f for f in collab_features_cols if f in features_to_use])}/{len(collab_features_cols)}")
print(f"   - TOTAL: {len(features_to_use)} features")

if train_missing == 0 and val_missing == 0 and y_train_missing == 0 and y_val_missing == 0:
    print("\n" + "🎉" * 50)
    print("✅ FEATURE MATRICES SUCCESSFULLY CREATED!")
    print("✅ NO MISSING VALUES!")
    print("✅ DATA SAVED TO FILES!")
    print("✅ READY FOR MODELING!")
    print("🎉" * 50)
else:
    print("\n⚠️  WARNING: Some missing values detected - review above")

print("\n📁 Files created:")
print("   - feature_matrices.pkl (all data in one file)")
print("   - X_train.csv, X_val.csv (feature matrices)")
print("   - y_train.csv, y_val.csv (target vectors)")


FINAL SUMMARY - FEATURE MATRICES READY FOR MODELING!

✅ TRAIN SET:
   - X_train: (6158, 19) (rows × features)
   - y_train: (6158,) (rows,)
   - Missing values: 0

✅ VALIDATION SET:
   - X_val: (1414, 19) (rows × features)
   - y_val: (1414,) (rows,)
   - Missing values: 0

📊 FEATURE BREAKDOWN:
   - Audio features: 7/7
   - Temporal features: 5/5
   - Artist features: 3/3
   - Collaboration features: 4/4
   - TOTAL: 19 features

🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉
✅ FEATURE MATRICES SUCCESSFULLY CREATED!
✅ NO MISSING VALUES!
✅ DATA SAVED TO FILES!
✅ READY FOR MODELING!
🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉🎉

📁 Files created:
   - feature_matrices.pkl (all data in one file)
   - X_train.csv, X_val.csv (feature matrices)
   - y_train.csv, y_val.csv (target vectors)
